In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from instruments.gilson.gilson_ethernet import GilsonEthernet
from instruments.gilson.tray import Tray
from instruments.gilson.rack import Rack_209, Rack_3dp
from core.logging import flow_logger as logger
from instruments.vici.dim import DIM
from instruments.vici.fsm import FSM
from instruments.syr_pumps.HB_syr_pump import HBElite
from instruments.knauer.knauer_pump_azura import KnauerPumpAzura
from ecosystems.reaction_segment_generation import RSG
from ecosystems.segmentation import Segmentation


# Experiment framework
from core.experiment_context import ExperimentManager
from core.experiment_builder import ExperimentBuilder
from core.experiment_validator import ExperimentValidator
from core.experiment_compiler import ExperimentCompiler
from core.experiment_intent import ExperimentIntent
from core.inventory import Inventory

In [2]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4


In [3]:
# --- Instantiate the tray ---
tray = Tray()

# --- Instantiate modules for the tray ---
rack1 = Rack_209()
rack2 = Rack_3dp()

# --- Assign modules to tray slots ---
tray.assign_slot(1, rack1, alias = "rack1")    # Assigns rack1 to slot 1
tray.assign_slot(2, rack2, alias = "rack2")    # Assigns rack2 to slot 2
tray.assign_slot(3, dim, alias = "dim")      # Assigns dim to slot 3

In [4]:
# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("VB-1E-7")

In [5]:
# Instantiate the RSG ecosystem with the connected Gilson, pump, and DIM
rsg = RSG(gilson=g, pump=syr_pump, dim=dim)

# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, fsm=fsm, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [11]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [6]:
inventory = Inventory()

# Clear + reassign (optional but safer for now)
inventory.assign(
    module="rack2",
    vial=1,
    name="MeCN",
    concentration_M=None,
    solvent="MeCN",
    current_volume_uL=80000,
    min_safe_volume_uL=20000
)

In [7]:
intent = ExperimentIntent(
    experiment_id="VB-1E-7",
    description="Single 100uL MeCN slug for VB-1E-7"
)

intent.add_block(
    name="Single MeCN slug",
    components=["MeCN"],
    ratios=[[100]],  
    total_volume_uL=100.0
)

intent.summary()

{'experiment_id': 'VB-1E-7',
 'description': 'Single 100uL MeCN slug for VB-1E-7',
 'num_blocks': 1,
 'estimated_slugs': 1,
 'blocks': [{'name': 'Single MeCN slug',
   'components': ['MeCN'],
   'num_slugs': 1,
   'total_volume_uL': 100.0,
   'fixed_total_volume': True}]}

In [8]:
compiler = ExperimentCompiler(inventory)

compiled_df = compiler.compile(intent)

compiled_df

,slug_id,module,vial,volume_uL,component,block_id,slug_order,component_order
0,block_1_slug_1,rack2,1,100.0,MeCN,block_1,1,1


In [9]:
builder = ExperimentBuilder(inventory=inventory)

result = builder.build_and_create(
    experiment_id="VB-1E-7",
    rows=compiled_df,
    description=intent.description,
    global_conditions={
        "flowrate_ul_min": 1000,
        "gas_prime_s": 20
    },
    overwrite=True
)

print(result["plan_path"])

pd.DataFrame(result["summary"])

C:\Users\CHAD-HPLC\Documents\VictorFlow\Experiments\VB-1E-7\plan.json


,slug_id,num_components,total_volume_uL,components
0,block_1_slug_1,1,100.0,"[(MeCN, rack2, 1, 100.0)]"


In [10]:
manager = ExperimentManager()

manager.load_experiment("VB-1E-7")

manager.status()

[EXPERIMENT] VB-1E-7 loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] State = prepared
[EXPERIMENT] Awaiting arm_experiment()
Mode: experiment
Experiment: VB-1E-7
State: prepared
System state: UNKNOWN
Reactor state: UNKNOWN
RSG state: UNKNOWN
Platform state: UNKNOWN
Slug index: 0


In [11]:
trace = manager.preview_execution(
    seg,
    wash_policy="none",
)

for step in trace:
    print(step)


--- SLUG 1: block_1_slug_1 ---
seg.prime_gas_path(20.0s)
seg.create_slug(...)
  → prepare_slug()
    → DIM.load()
    → RSG.build_rxn_segment()
      → pickup 100.0 uL from rack2 vial 1
    → post-pickup air gap (5.0 uL)
    → dispense to DIM
  → launch_segment()
    → FSM.carrier_to_dim()
    → DIM.inject()
    → carrier.set_flow_rate(1000.0)
    → carrier.start_flow()
    → phase transition: LOOP_LOADED → RUNNING


In [ ]:
syr_pump.infuse_volume(20, 0.2)

In [12]:
manager.precondition_reactor(seg, carrier_flowrate_ul_min=1000, stabilisation_time_s=60)

manager.prepare_rsg(
    seg,
    air_gap=20.0,
    rate_wdr=0.10
)

[REACTOR] Setting valve positions
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started
[REACTOR] Stabilising...
[SYSTEM] Reactor READY
[RSG] Initialising
[Initialise] Setting up start condition
[Air Gap] 20.0uL @ 0.1mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Initialise] Ready - air gap in place
[SYSTEM] RSG READY


In [13]:
manager.arm_experiment()

[EXPERIMENT] VB-1E-7 armed
[EXPERIMENT] Awaiting execute_experiment()


In [17]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [16]:
g.go_to_dim()

ValueError: No module assigned to slot dim.

In [ ]:
manager.execute_experiment(
    seg,
    wash_policy="none",
)